In [11]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv

load_dotenv("/Users/bytedance/Documents/Mauriciio/Jupyter/Metallica/.env")

LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")
BASE_URL = "http://ws.audioscrobbler.com/2.0/"

big4_tracks = pd.read_csv("big4_studio_tracks_musicbrainz.csv")

big4_tracks.head()

,artist,album_name,first_release_date,release_group_id,release_id,recording_id,track_number,track_position,track_name,track_length_ms
0,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,a620f5e7-ae9d-4089-a947-a1cee4a9435e,A1,1,Hit the Lights,257000.0
1,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,04fc57f2-4dc8-4b84-801b-3699508597e8,A2,2,The Four Horsemen,432706.0
2,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,65ca9607-fa5d-4b5d-b830-3fb168ac9064,A3,3,Motorbreath,188133.0
3,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,321b3122-f443-4709-8098-3afba7cddca9,A4,4,Jump in the Fire,281493.0
4,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,78095e21-30cf-47eb-b367-3af6ee426af5,A5,5,(Anesthesia) Pulling Teeth,254840.0


In [12]:
## Tracks listeners
def get_lastfm_track_info(artist, track):
    params = {
        "method": "track.getInfo",
        "artist": artist,
        "track": track,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1
    }
    
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    
    if "track" not in data:
        return {
            "lastfm_listeners": None,
            "lastfm_playcount": None,
            "lastfm_url": None
        }
    
    track_data = data["track"]
    
    return {
        "lastfm_listeners": int(track_data.get("listeners", 0)),
        "lastfm_playcount": int(track_data.get("playcount", 0)),
        "lastfm_url": track_data.get("url")
    }

In [13]:
lastfm_rows = []

for index, row in big4_tracks.iterrows():
    artist = row["artist"]
    track = row["track_name"]
    
    print(f"{index + 1}/{len(big4_tracks)} - {artist} - {track}")
    
    info = get_lastfm_track_info(artist, track)
    
    lastfm_rows.append({
        "artist": artist,
        "track_name": track,
        **info
    })
    
    time.sleep(0.25)

1/665 - Metallica - Hit the Lights
2/665 - Metallica - The Four Horsemen
3/665 - Metallica - Motorbreath
4/665 - Metallica - Jump in the Fire
5/665 - Metallica - (Anesthesia) Pulling Teeth
6/665 - Metallica - Whiplash
7/665 - Metallica - Phantom Lord
8/665 - Metallica - No Remorse
9/665 - Metallica - Seek & Destroy
10/665 - Metallica - Metal Militia
11/665 - Metallica - Fight Fire With Fire
12/665 - Metallica - Ride the Lightning
13/665 - Metallica - For Whom the Bell Tolls
14/665 - Metallica - Fade to Black
15/665 - Metallica - Trapped Under Ice
16/665 - Metallica - Escape
17/665 - Metallica - Creeping Death
18/665 - Metallica - The Call of Ktulu
19/665 - Metallica - Battery
20/665 - Metallica - Master of Puppets
21/665 - Metallica - The Thing That Should Not Be
22/665 - Metallica - Welcome Home (Sanitarium)
23/665 - Metallica - Disposable Heroes
24/665 - Metallica - Leper Messiah
25/665 - Metallica - Orion
26/665 - Metallica - Damage, Inc.
27/665 - Metallica - Blackened
28/665 - Meta

In [14]:
lastfm_df = pd.DataFrame(lastfm_rows)

big4_enriched = big4_tracks.merge(
    lastfm_df,
    on=["artist", "track_name"],
    how="left"
)

big4_enriched.head()


,artist,album_name,first_release_date,release_group_id,release_id,recording_id,track_number,track_position,track_name,track_length_ms,lastfm_listeners,lastfm_playcount,lastfm_url
0,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,a620f5e7-ae9d-4089-a947-a1cee4a9435e,A1,1,Hit the Lights,257000.0,397876,2596232,https://www.last.fm/music/Metallica/_/Hit+the+...
1,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,a620f5e7-ae9d-4089-a947-a1cee4a9435e,A1,1,Hit the Lights,257000.0,397876,2596232,https://www.last.fm/music/Metallica/_/Hit+the+...
2,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,04fc57f2-4dc8-4b84-801b-3699508597e8,A2,2,The Four Horsemen,432706.0,400450,2786243,https://www.last.fm/music/Metallica/_/The+Four...
3,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,04fc57f2-4dc8-4b84-801b-3699508597e8,A2,2,The Four Horsemen,432706.0,400450,2786243,https://www.last.fm/music/Metallica/_/The+Four...
4,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,65ca9607-fa5d-4b5d-b830-3fb168ac9064,A3,3,Motorbreath,188133.0,394871,2717223,https://www.last.fm/music/Metallica/_/Motorbreath


In [17]:
big4_enriched.duplicated(
    subset=["artist", "album_name", "track_name"]
).sum()

np.int64(161)

In [18]:
dups = big4_enriched[
    big4_enriched.duplicated(
        subset=["artist", "album_name", "track_name"],
        keep=False
    )
]

dups.sort_values(["artist", "album_name", "track_name"])dups[[
    "artist",
    "album_name",
    "track_name",
    "track_length_ms",
    "lastfm_playcount",
    "lastfm_listeners"
]].sort_values(["artist", "album_name", "track_name"])

,artist,album_name,first_release_date,release_group_id,release_id,recording_id,track_number,track_position,track_name,track_length_ms,lastfm_listeners,lastfm_playcount,lastfm_url
648,Anthrax,Among the Living,1987-03-22,413bc69f-9391-3fe3-802b-5bf4922f6066,fbc164e6-c628-4a73-b20d-063c1e335408,0e511421-ee06-47df-ba0d-a58163fdf7a0,1,1,Among the Living,316200.0,222169,1134213,https://www.last.fm/music/Anthrax/_/Among+the+...
649,Anthrax,Among the Living,1987-03-22,413bc69f-9391-3fe3-802b-5bf4922f6066,fbc164e6-c628-4a73-b20d-063c1e335408,0e511421-ee06-47df-ba0d-a58163fdf7a0,1,1,Among the Living,316200.0,222169,1134213,https://www.last.fm/music/Anthrax/_/Among+the+...
650,Anthrax,Among the Living,1987-03-22,413bc69f-9391-3fe3-802b-5bf4922f6066,fbc164e6-c628-4a73-b20d-063c1e335408,732168d4-c45a-4713-b4a5-7049cdb995e6,2,2,Caught in a Mosh,300133.0,339098,2040366,https://www.last.fm/music/Anthrax/_/Caught+in+...
651,Anthrax,Among the Living,1987-03-22,413bc69f-9391-3fe3-802b-5bf4922f6066,fbc164e6-c628-4a73-b20d-063c1e335408,732168d4-c45a-4713-b4a5-7049cdb995e6,2,2,Caught in a Mosh,300133.0,339098,2040366,https://www.last.fm/music/Anthrax/_/Caught+in+...
652,Anthrax,Among the Living,1987-03-22,413bc69f-9391-3fe3-802b-5bf4922f6066,fbc164e6-c628-4a73-b20d-063c1e335408,732168d4-c45a-4713-b4a5-7049cdb995e6,2,2,Caught in a Mosh,300133.0,339098,2040366,https://www.last.fm/music/Anthrax/_/Caught+in+...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
439,Slayer,South of Heaven,1988-07-05,74292121-e345-3f10-85ae-013f4e5a8cae,90102c45-9bc6-4a6e-9a13-0d0b9c0f877c,95c4a3af-cdc3-4716-9e70-3ec0eff03359,A5,5,Mandatory Suicide,245133.0,288187,1816952,https://www.last.fm/music/Slayer/_/Mandatory+S...
433,Slayer,South of Heaven,1988-07-05,74292121-e345-3f10-85ae-013f4e5a8cae,90102c45-9bc6-4a6e-9a13-0d0b9c0f877c,5659354b-92af-4078-8287-c9ab59501994,A1,1,South of Heaven,298466.0,551169,4105344,https://www.last.fm/music/Slayer/_/South+of+He...
434,Slayer,South of Heaven,1988-07-05,74292121-e345-3f10-85ae-013f4e5a8cae,90102c45-9bc6-4a6e-9a13-0d0b9c0f877c,5659354b-92af-4078-8287-c9ab59501994,A1,1,South of Heaven,298466.0,551169,4105344,https://www.last.fm/music/Slayer/_/South+of+He...
527,Slayer,World Painted Blood,2009-10-28,1722d206-eba4-426e-8977-599057ccdbc8,99241fe7-0e0b-4544-9db4-0071e863d194,6708235b-b4aa-42b7-9b37-47525b04ace9,5,5,Hate Worldwide,172026.0,106944,741155,https://www.last.fm/music/Slayer/_/Hate+Worldwide


In [19]:
dups[[
    "artist",
    "album_name",
    "track_name",
    "track_length_ms",
    "lastfm_playcount",
    "lastfm_listeners"
]].sort_values(["artist", "album_name", "track_name"])

,artist,album_name,track_name,track_length_ms,lastfm_playcount,lastfm_listeners
648,Anthrax,Among the Living,Among the Living,316200.0,1134213,222169
649,Anthrax,Among the Living,Among the Living,316200.0,1134213,222169
650,Anthrax,Among the Living,Caught in a Mosh,300133.0,2040366,339098
651,Anthrax,Among the Living,Caught in a Mosh,300133.0,2040366,339098
652,Anthrax,Among the Living,Caught in a Mosh,300133.0,2040366,339098
...,...,...,...,...,...,...
439,Slayer,South of Heaven,Mandatory Suicide,245133.0,1816952,288187
433,Slayer,South of Heaven,South of Heaven,298466.0,4105344,551169
434,Slayer,South of Heaven,South of Heaven,298466.0,4105344,551169
527,Slayer,World Painted Blood,Hate Worldwide,172026.0,741155,106944


In [20]:
big4_enriched_clean = big4_enriched.drop_duplicates(
    subset=["artist", "album_name", "track_name"],
    keep="first"
).copy()

In [21]:
print("Before:", big4_enriched.shape)
print("After:", big4_enriched_clean.shape)

big4_enriched_clean.duplicated(
    subset=["artist", "album_name", "track_name"]
).sum()

Before: (813, 13)
After: (652, 13)


np.int64(0)

In [22]:

## Save to csv
big4_enriched_clean.to_csv("big4_tracks_enriched_lastfm.csv", index=False)